# Pipeline test

In [1]:
import pandas as pd
import numpy as np
from feature_engine.encoding import MeanEncoder,OneHotEncoder,RareLabelEncoder,CountFrequencyEncoder
from feature_engine.imputation import AddMissingIndicator
from feature_engine.selection import DropFeatures
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.utils.validation import check_is_fitted
from sklearn.pipeline import Pipeline
from dataclasses import dataclass
import duckdb
#helping to make 
from typing import Optional, Literal, Dict, Any
import sys
sys.path.append('../src')
from py_def_class import Add_Column,RFPermutationRegressorSelector

In [2]:
df = pd.read_csv(r'..\data\processed\Cleaned_quali_f1.csv')
#dropping the track temperature columns
filter_out = [fo for fo in df.columns if 'TrackTemp' in fo]
df = df.drop(filter_out,axis=1)
df = df[df['laptime_sum_sectortimes_quali'].notna()].copy()
df.head()

,Driver,Team,GP,Rule_Era,AirTemp_p1,Humidity_p1,Pressure_p1,Rainfall_p1,laptime_sum_sectortimes_p1,LapTimeDiff_p1,...,Direction,Circut_length,Turns,Pace_profile,Flat_out_run,Slow_turns,Medium_turns,High_speed_turns,Turn_density,Complexity_label
0,VER,Red Bull Racing,Abu Dhabi GP,High-Downforce,29.7,33.0,1015.7,0.0,98.491,0.000,...,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced
1,RIC,Red Bull Racing,Abu Dhabi GP,High-Downforce,29.9,29.4,1015.5,0.0,98.945,0.454,...,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced
2,BOT,Mercedes,Abu Dhabi GP,High-Downforce,29.7,32.6,1015.7,0.0,99.452,0.961,...,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced
3,HAM,Mercedes,Abu Dhabi GP,High-Downforce,29.6,33.2,1015.8,0.0,99.543,1.052,...,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced
4,OCO,Racing Point,Abu Dhabi GP,High-Downforce,29.7,32.8,1015.8,0.0,100.102,1.611,...,Anti-clockwise,5.28,16,Balanced-high speed,1120,5,6,5,3.03,Balanced


In [3]:
X = df.drop(['laptime_sum_sectortimes_quali'],axis=1).copy()
y = df['laptime_sum_sectortimes_quali'].copy()

In [4]:
df.select_dtypes(include='O').head()

,Driver,Team,GP,Rule_Era,Compound_p1,Compound_p2,Compound_p3,Sprint-Session,Compound_sprint_quali,Session,Compound_quali,Sprint_Race_Era,Team_Lineage,Type,Direction,Pace_profile,Complexity_label
0,VER,Red Bull Racing,Abu Dhabi GP,High-Downforce,HYPERSOFT,HYPERSOFT,HYPERSOFT,NaN,NaN,Q3,HYPERSOFT,2023-2026,Red Bull Racing,Race,Anti-clockwise,Balanced-high speed,Balanced
1,RIC,Red Bull Racing,Abu Dhabi GP,High-Downforce,HYPERSOFT,HYPERSOFT,HYPERSOFT,NaN,NaN,Q3,HYPERSOFT,2023-2026,Red Bull Racing,Race,Anti-clockwise,Balanced-high speed,Balanced
2,BOT,Mercedes,Abu Dhabi GP,High-Downforce,SUPERSOFT,HYPERSOFT,HYPERSOFT,NaN,NaN,Q3,HYPERSOFT,2023-2026,Mercedes,Race,Anti-clockwise,Balanced-high speed,Balanced
3,HAM,Mercedes,Abu Dhabi GP,High-Downforce,ULTRASOFT,HYPERSOFT,HYPERSOFT,NaN,NaN,Q3,HYPERSOFT,2023-2026,Mercedes,Race,Anti-clockwise,Balanced-high speed,Balanced
4,OCO,Racing Point,Abu Dhabi GP,High-Downforce,HYPERSOFT,HYPERSOFT,HYPERSOFT,NaN,NaN,Q3,HYPERSOFT,2023-2026,Force India-Racing Point-Aston Martin,Race,Anti-clockwise,Balanced-high speed,Balanced


In [5]:
#list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
missing_in_sprint_we = [sprint_not for sprint_not in df.columns if 'p2' in sprint_not or 'p3' in sprint_not]
#
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

#tol => percentage/frequency of appearance
test_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=0.0019,replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=0.01,replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=0.015,
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(
        col=mean_freq,
        name_= ['_mean','_frequency']
    )),
    ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore')),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore')),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=0.1))
]).fit_transform(X,y)
test_pipeline.columns

c:\Users\gandj\Documents\F1\Manual\f1_venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:260: UserWarning: During the encoding, NaN values were introduced in the feature(s) Compound_p1_mean, Compound_p2_mean, Compound_p3_mean, Compound_sprint_quali_mean, Compound_quali_mean, Sprint-Session_mean.
  warnings.warn(
c:\Users\gandj\Documents\F1\Manual\f1_venv\Lib\site-packages\feature_engine\encoding\base_encoder.py:260: UserWarning: During the encoding, NaN values were introduced in the feature(s) Compound_p1_frequency, Compound_p2_frequency, Compound_p3_frequency, Compound_sprint_quali_frequency, Compound_quali_frequency, Sprint-Session_frequency.
  warnings.warn(


Index(['laptime_sum_sectortimes_p1', 'laptime_sum_sectortimes_p2',
       'laptime_sum_sectortimes_p3', 'GP_mean', 'Compound_quali_mean'],
      dtype='object')

In [6]:
list(test_pipeline.columns)

['laptime_sum_sectortimes_p1',
 'laptime_sum_sectortimes_p2',
 'laptime_sum_sectortimes_p3',
 'GP_mean',
 'Compound_quali_mean']

In [7]:
len(test_pipeline.columns)

5

In [8]:
len(df.columns)

58

In [9]:
test_pipeline.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3543 entries, 0 to 3784
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   laptime_sum_sectortimes_p1  3309 non-null   float64
 1   laptime_sum_sectortimes_p2  3048 non-null   float64
 2   laptime_sum_sectortimes_p3  2895 non-null   float64
 3   GP_mean                     3543 non-null   float64
 4   Compound_quali_mean         3542 non-null   float64
dtypes: float64(5)
memory usage: 166.1 KB
